In [ ]:
import random
import json
from collections import defaultdict

class BlackjackEnv:
    def __init__(self):
        self.action_space = [0, 1] 
        self.cards = [2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 10, 10, 11]

    def _draw_card(self):
        return random.choice(self.cards)

    def _calculate_score(self, hand):
        score = sum(hand)
        aces = hand.count(11)
        while score > 21 and aces > 0:
            score -= 10
            aces -= 1
        return score

    def _has_usable_ace(self, hand):
        return 11 in hand and sum(hand) <= 21

    def _get_obs(self):
        player_score = self._calculate_score(self.player_hand)
        dealer_card = self.dealer_hand[0]
        usable_ace = self._has_usable_ace(self.player_hand)
        return (player_score, dealer_card, usable_ace)

    def reset(self):
        self.player_hand = [self._draw_card(), self._draw_card()]
        self.dealer_hand = [self._draw_card(), self._draw_card()]
        return self._get_obs()

    def step(self, action):
        if action == 1:  
            self.player_hand.append(self._draw_card())
            if self._calculate_score(self.player_hand) > 21:
                return self._get_obs(), -1.0, True 
            else:
                return self._get_obs(), 0.0, False  
                
        elif action == 0:  
            while self._calculate_score(self.dealer_hand) < 17:
                self.dealer_hand.append(self._draw_card())
                
            player_score = self._calculate_score(self.player_hand)
            dealer_score = self._calculate_score(self.dealer_hand)

            if dealer_score > 21:
                reward = 1.0  
            elif player_score > dealer_score:
                reward = 1.0  
            elif player_score < dealer_score:
                reward = -1.0 
            else:
                reward = 0.0  

            return self._get_obs(), reward, True

def train_agent(episodes=500000):
    env = BlackjackEnv()
    
    q_table = defaultdict(lambda: [0.0, 0.0])

    alpha = 0.05        
    gamma = 1.0         
    epsilon = 1.0       
    epsilon_decay = 0.99999
    min_epsilon = 0.05  

    print(f"Rozpoczynam trening na {episodes} epizodach...")

    for i in range(episodes):
        state = env.reset()
        done = False

        while not done:
            if random.random() < epsilon:
                action = random.choice([0, 1]) 
            else:
                action = 0 if q_table[state][0] > q_table[state][1] else 1

            next_state, reward, done = env.step(action)

            old_value = q_table[state][action]
            next_max = max(q_table[next_state]) if not done else 0.0
            
            new_value = old_value + alpha * (reward + gamma * next_max - old_value)
            q_table[state][action] = new_value

            state = next_state

        epsilon = max(min_epsilon, epsilon * epsilon_decay)

        if (i + 1) % 100000 == 0:
            print(f"Ukończono {i + 1} epizodów...")

    return q_table

if __name__ == "__main__":
    trained_q_table = train_agent(episodes=1_000_000)

    json_q_table = {}
    for state, actions in trained_q_table.items():
        state_str = f"{state[0]}_{state[1]}_{state[2]}"
        json_q_table[state_str] = actions

    with open("blackjack_bot_brain.json", "w") as f:
        json.dump(json_q_table, f, indent=4)

    print("Trening zakończony! Zapisano mózg bota w pliku 'blackjack_bot_brain.json'.")

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class GymBlackjackEnv(gym.Env):
    def __init__(self):
        super().__init__()
        
        # Akcje: 0 (Stand), 1 (Hit), 2 (Double)
        self.action_space = spaces.Discrete(3)
        self.cards = [2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 10, 10, 11]
        
        # Obserwacje: (suma_gracza, karta_krupiera, uzyteczny_as)
        self.observation_space = spaces.Box(
            low=np.array([4, 1, 0]), 
            high=np.array([31, 11, 1]), 
            dtype=np.float32
        )
    
    def _draw_card(self):
        return random.choice(self.cards)
    
    def _calculate_score(self, hand):
        score = sum(hand)
        aces = hand.count(11)
        while score > 21 and aces > 0:
            score -= 10
            aces -= 1
        return score

    def _has_usable_ace(self, hand):
        return 11 in hand and sum(hand) <= 21
    
    def _get_obs(self):
        """
        Przekształca wewnętrzny stan gry na wektor (tablicę) zrozumiałą dla sieci neuronowej.
        """
        player_score = self._calculate_score(self.player_hand)
        dealer_card = self.dealer_hand[0] 
        
        usable_ace = int(self._has_usable_ace(self.player_hand)) 
        
        return np.array([player_score, dealer_card, usable_ace], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.player_hand = [self._draw_card(), self._draw_card()]
        self.dealer_hand = [self._draw_card(), self._draw_card()]
        
        obs = self._get_obs()
        
        info = {}
        
        return obs, info

    def step(self, action):
        reward = 0.0
        terminated = False
        truncated = False
        info = {}
        if action == 2:  
            self.player_hand.append(self._draw_card())
            terminated = True
            if self._calculate_score(self.player_hand) > 21:
                reward = -2.0
        elif action == 1:
            self.player_hand.append(self._draw_card())
            if self._calculate_score(self.player_hand) > 21:
                reward = -1.0  
                terminated = True
        elif action == 0:
            terminated = True

        player_score = self._calculate_score(self.player_hand)
        if terminated and player_score <= 21:

            while self._calculate_score(self.dealer_hand) < 17:
                self.dealer_hand.append(self._draw_card())
            dealer_score = self._calculate_score(self.dealer_hand)
            multiplier = 2.0 if action == 2 else 1.0  

            if dealer_score > 21:
                reward = 1.0 * multiplier  
            elif player_score > dealer_score:
                reward = 1.0 * multiplier  
            elif player_score < dealer_score:
                reward = -1.0 * multiplier 
            else:
                reward = 0.0 

           
        
        if terminated:
            info['final_player_hand'] = self.player_hand
            info['final_dealer_hand'] = self.dealer_hand
        
        return self._get_obs(), reward, terminated, truncated, info

In [ ]:
from stable_baselines3 import DQN

env = GymBlackjackEnv()

# 2. Inicjalizujesz model (MlpPolicy oznacza wielowarstwową sieć neuronową)
# model = DQN("MlpPolicy", 
#             env, 
#             verbose=1,
#             device="cpu", 
#             buffer_size=10000,
#             exploration_fraction=0.4,
#             optimize_memory_usage=True,
#             replay_buffer_kwargs={"handle_timeout_termination": False})

model = DQN(
    "MlpPolicy", 
    env, 
    verbose=1,
    learning_rate=5e-4,          
    buffer_size=50000,
    learning_starts=2000, 
    batch_size=64,               
    exploration_fraction=0.4,    
    exploration_final_eps=0.02,  
    target_update_interval=1000, 
    replay_buffer_kwargs={"handle_timeout_termination": False}
)

model.learn(total_timesteps=500000, progress_bar=True)

model.save("dqn_blackjack_bot")

# Wyniki bez możliwości podwajania
## === WYNIKI TESTU ===

Wygrane gracza: 4224 (42.24%)  
Przegrane (Wygrane kasyna): 4769 (47.69%)  
Remisy: 1007 (10.07%)  
Średnia nagroda na partię: -0.0545  
(42.24, 47.69, 10.07) 

## Wyniki z Podwajaniem

=== WYNIKI TESTU ===  
Rozegrane partie: 10000  
Wygrane gracza: 4221 (42.21%)  
Przegrane (Wygrane kasyna): 4821 (48.21%)  
Remisy: 958 (9.58%)  
Średnia nagroda na partię: -0.0600  
(42.21, 48.209999999999994, 9.58)  

In [ ]:
def evaluate_blackjack_model(model, env, num_episodes=10000):
    wins = 0
    losses = 0
    draws = 0
    total_reward = 0.0


    for _ in range(num_episodes):
        obs, info = env.reset()
        done = False
        
        while not done:
            action, _states = model.predict(obs, deterministic=True)
            
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

        total_reward += reward
        if reward > 0:
            wins += 1
        elif reward < 0:
            losses += 1
        else:
            draws += 1

    win_rate = (wins / num_episodes) * 100
    loss_rate = (losses / num_episodes) * 100
    draw_rate = (draws / num_episodes) * 100
    avg_reward = total_reward / num_episodes

    print("\n=== WYNIKI TESTU ===")
    print(f"Rozegrane partie: {num_episodes}")
    print(f"Wygrane gracza: {wins} ({win_rate:.2f}%)")
    print(f"Przegrane (Wygrane kasyna): {losses} ({loss_rate:.2f}%)")
    print(f"Remisy: {draws} ({draw_rate:.2f}%)")
    print(f"Średnia nagroda na partię: {avg_reward:.4f}")
    
    return win_rate, loss_rate, draw_rate

evaluate_blackjack_model(model, env, num_episodes=10000)

In [ ]:
import pandas as pd

def analyze_bot_strategy(model):
    player_sums = range(8, 20)
    dealer_cards = range(2, 12) 
    usable_ace = 0 

    strategy_map = []

    for player_score in player_sums:
        row = []
        for dealer_card in dealer_cards:
            obs = np.array([player_score, dealer_card, usable_ace], dtype=np.float32)
            
            action, _ = model.predict(obs, deterministic=True)
            
            action_symbol = {0: "S", 1: "H", 2: "D"}.get(int(action))
            row.append(action_symbol)
        strategy_map.append(row)

    df = pd.DataFrame(
        strategy_map, 
        columns=[f"D:{c}" for c in dealer_cards], 
        index=[f"P:{s}" for s in player_sums]
    )
    
    print("\n=== TABELA STRATEGII BOTA (S=Stand, H=Hit, D=Double) ===")
    print("Analiza dla rąk bez użytecznego asa:")
    print(df)
    return df

analyze_bot_strategy(model)